In [ ]:
!pip install hmmlearn

In [ ]:
# --- 1. Load and Prepare Data ---
try:
    # Assuming 'data.csv' is in the same directory
    df = pd.read_csv("data.csv")
except FileNotFoundError:
    print("Error: 'data.csv' not found. Please ensure the file is in the correct path.")
    # Create dummy data for demonstration if file isn't found
    data = {
        'Date': pd.to_datetime(pd.date_range(start='2023-01-01', periods=100)),
        'Close': 100 + np.cumsum(np.random.randn(100) * 5),
        'News - Positive Sentiment': np.random.rand(100),
        'News - Negative Sentiment': np.random.rand(100),
        'News - New Products': np.random.randint(0, 5, 100),
        'News - Layoffs': np.random.randint(0, 3, 100),
        'News - Analyst Comments': np.random.randint(0, 10, 100),
        'News - Stocks': np.random.randint(0, 5, 100),
        'News - Dividends': np.random.randint(0, 2, 100),
        'News - Corporate Earnings': np.random.randint(0, 5, 100),
        'News - Mergers & Acquisitions': np.random.randint(0, 2, 100),
        'News - Store Openings': np.random.randint(0, 3, 100),
        'News - Product Recalls': np.random.randint(0, 2, 100),
        'News - Adverse Events': np.random.randint(0, 2, 100),
        'News - Personnel Changes': np.random.randint(0, 5, 100),
        'News - Stock Rumors': np.random.randint(0, 2, 100)
    }
    df = pd.DataFrame(data)
    print("Using dummy data for execution.")

df['Date'] = pd.to_datetime(df['Date'])
# Target variable: next day's price change (Close[t+1] - Close[t])
df['Price_Movement'] = df['Close'].shift(-1) - df['Close']
df.dropna(inplace=True)

df.head()


,Date,Open,High,Low,Close,Adj Close,Volume,Symbol,Security,GICS Sector,...,News - Stocks,News - Dividends,News - Corporate Earnings,News - Mergers & Acquisitions,News - Store Openings,News - Product Recalls,News - Adverse Events,News - Personnel Changes,News - Stock Rumors,Price_Movement
1,2020-10-01,160.669998,161.899994,157.720001,158.789993,149.612045,1989100.0,MMM,3M,Industrials,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.570007
2,2020-10-02,156.470001,161.940002,156.250000,160.360001,151.091309,1768600.0,MMM,3M,Industrials,...,2.0,0.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0,2.389999
3,2020-10-05,162.250000,163.500000,161.759995,162.750000,153.343170,1457000.0,MMM,3M,Industrials,...,2.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,-0.520004
4,2020-10-06,163.440002,165.699997,161.830002,162.229996,152.853195,2021900.0,MMM,3M,Industrials,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,4.260010
5,2020-10-07,165.100006,167.750000,164.410004,166.490005,156.866989,2155400.0,MMM,3M,Industrials,...,17.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.220001


In [ ]:
features = [
    'News - Positive Sentiment', 'News - Negative Sentiment', 
    'News - New Products', 'News - Layoffs', 'News - Analyst Comments',
    'News - Stocks', 'News - Dividends', 'News - Corporate Earnings',
    'News - Mergers & Acquisitions', 'News - Store Openings', 
    'News - Product Recalls', 'News - Adverse Events', 
    'News - Personnel Changes', 'News - Stock Rumors'
]

X = df[features]
y = df['Price_Movement']

# --- 2. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

subset = min(200, len(y_test))  # For visualization
print("Training and testing data prepared successfully.")


Training and testing data prepared successfully.


In [ ]:
# --- 3. Model Training and Prediction ---

# Linear Regression
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lin = lin_model.predict(X_test)

# Support Vector Regression (SVR)
svr_model = SVR(kernel='rbf', C=10, epsilon=0.1)
svr_model.fit(X_train, y_train)
y_pred_svr = svr_model.predict(X_test)

# Bayesian Ridge Regression
bayesian_model = BayesianRidge()
bayesian_model.fit(X_train, y_train)
y_pred_bayesian = bayesian_model.predict(X_test)

print("✅ All models trained and predictions generated.")


In [ ]:
# --- 4. Model Evaluation ---

rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred_lin))
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred_svr))
rmse_bayesian = np.sqrt(mean_squared_error(y_test, y_pred_bayesian))

print("---------------------------------------")
print("--- Model Performance (RMSE: Lower is Better) ---")
print(f"Linear Regression RMSE: {rmse_lin:.4f}")
print(f"SVR (RBF) RMSE: {rmse_svr:.4f}")
print(f"Bayesian Ridge RMSE: {rmse_bayesian:.4f}")
print("---------------------------------------")


In [ ]:
plt.figure(figsize=(14, 7))

# Actual vs Predicted
plt.plot(y_test[:subset].values, label='Actual Price Movement', color='black', linewidth=2)
plt.plot(y_pred_lin[:subset], label='Linear Regression Predicted', linestyle='--', alpha=0.7)
plt.plot(y_pred_svr[:subset], label='SVR (RBF) Predicted', linestyle=':', alpha=0.8)
plt.plot(y_pred_bayesian[:subset], label='Bayesian Ridge Predicted', linestyle='-.', alpha=0.7)

plt.title(f'Comparison of Models: Actual vs. Predicted Price Movement (First {subset} Test Days)')
plt.xlabel('Test Sample Index (Day)')
plt.ylabel('Price Movement (USD)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.axhline(0, color='gray', linestyle='-')
plt.show()


In [ ]:
# --- 6. Feature Importance Visualization ---
coefficients = lin_model.coef_
feature_names = X_train.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
importance_df['Absolute_Coefficient'] = importance_df['Coefficient'].abs()
importance_df = importance_df.sort_values(by='Absolute_Coefficient', ascending=True)

plt.figure(figsize=(12, 8))
colors = ['green' if c > 0 else 'red' for c in importance_df['Coefficient']]
plt.barh(importance_df['Feature'], importance_df['Coefficient'], color=colors)

plt.xlabel('Coefficient Value (Impact on Price Movement)')
plt.title('Feature Importance (Coefficients) from Linear Regression')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.axvline(0, color='black', linewidth=0.5)
plt.show()


In [ ]:
results = {
    'Linear Regression RMSE': rmse_lin,
    'SVR (RBF) RMSE': rmse_svr,
    'Bayesian Ridge RMSE': rmse_bayesian
}
pd.DataFrame([results]).to_csv('model_performance.csv', index=False)
print("Model performance saved as 'model_performance.csv'")
